In [1]:
# === 山峰最近兩個C0氣象站精確計算 ===
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, asin

def distance_haversine(lat1, lon1, lat2, lon2):
    """精確 Haversine 距離計算（公里）"""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 6371 * 2 * asin(sqrt(a))

# 載入資料
mountains = pd.read_csv('./csv/mountain_lanlon.csv').dropna()
stations = pd.read_csv('./csv/observation_stations.csv')

# 篩選 C0 開頭的氣象站並清理
c0_stations = stations[stations['站號'].str.startswith('C0', na=False)].copy()
c0_stations = c0_stations.dropna(subset=['經度', '緯度'])
c0_stations[['經度', '緯度']] = c0_stations[['經度', '緯度']].apply(pd.to_numeric, errors='coerce')
c0_stations = c0_stations.dropna(subset=['經度', '緯度'])

# *** 關鍵修正：重置索引 ***
c0_stations = c0_stations.reset_index(drop=True)

print(f"山峰數量: {len(mountains)}")
print(f"C0氣象站數量: {len(c0_stations)}")

# 計算結果
results = []

for _, mountain in mountains.iterrows():
    if pd.notna(mountain['山名']) and mountain['山名'].strip():
        mountain_name = mountain['山名']
        mountain_lat = mountain['緯度']
        mountain_lon = mountain['經度']
        
        # 追蹤最近的兩個氣象站
        first_distance = float('inf')
        second_distance = float('inf')
        first_station = None
        second_station = None
        
        # 對每個C0氣象站計算距離
        for idx, station in c0_stations.iterrows():
            distance = distance_haversine(
                mountain_lat, mountain_lon,
                station['緯度'], station['經度']
            )
            
            if distance < first_distance:
                # 新的最近距離，原本的最近變成第二近
                second_distance = first_distance
                second_station = first_station
                first_distance = distance
                first_station = station
            elif distance < second_distance:
                # 新的第二近距離
                second_distance = distance
                second_station = station
        
        result = {
            'mountain_name': mountain_name,
            'first_station_name': first_station['站名'],
            'first_station_id': first_station['站號'],
            'first_distance_km': round(first_distance, 3),
            'second_station_name': second_station['站名'] if second_station is not None else '',
            'second_station_id': second_station['站號'] if second_station is not None else '',
            'second_distance_km': round(second_distance, 3) if second_distance != float('inf') else '',
            'mountain_lat': mountain_lat,
            'mountain_lon': mountain_lon,
            'first_station_lat': first_station['緯度'],
            'first_station_lon': first_station['經度'],
            'first_station_elevation': first_station['海拔高度(m)'],
            'first_station_city': first_station['城市'],
            'second_station_lat': second_station['緯度'] if second_station is not None else '',
            'second_station_lon': second_station['經度'] if second_station is not None else '',
            'second_station_elevation': second_station['海拔高度(m)'] if second_station is not None else '',
            'second_station_city': second_station['城市'] if second_station is not None else ''
        }
        
        results.append(result)
        
        second_info = f"{second_station['站名']} ({second_distance:6.3f} km)" if second_station is not None else "無第二近"
        print(f"{mountain_name:15s} -> 1st: {first_station['站名']:15s} ({first_distance:6.3f} km), 2nd: {second_info}")

# 轉換為 DataFrame
df_results = pd.DataFrame(results)

# 保存結果
df_results.to_csv('csv/mountain_two_nearest.csv', index=False, encoding='utf-8-sig')

# 簡化版結果
simple = df_results[['mountain_name', 'first_station_name', 'first_station_id', 'first_distance_km', 
                     'second_station_name', 'second_station_id', 'second_distance_km']].copy()
simple.columns = ['山名', '最近氣象站', '站號', '距離1(公里)', '第二近氣象站', '站號2', '距離2(公里)']
simple.to_csv('csv/mountain_two_simple.csv', index=False, encoding='utf-8-sig')

# 統計資訊
print(f"\n統計:")
print(f"處理山峰: {len(df_results)} 座")
print(f"最近站平均距離: {df_results['first_distance_km'].mean():.3f} km")
print(f"第二近站平均距離: {df_results['second_distance_km'].replace('', np.nan).astype(float).mean():.3f} km")

print("\n結果已保存至:")
print("- csv/mountain_two_nearest.csv (詳細資料)")
print("- csv/mountain_two_simple.csv (簡化版)")


山峰數量: 15
C0氣象站數量: 504
奇萊主峰            -> 1st: 奇萊稜線            ( 2.604 km), 2nd: 小奇萊 ( 5.803 km)
奇萊北峰            -> 1st: 奇萊稜線            ( 1.288 km), 2nd: 小奇萊 ( 4.278 km)
桃山              -> 1st: 桃山              ( 0.112 km), 2nd: 雪山東峰 ( 6.098 km)
喀拉業山            -> 1st: 桃山              ( 2.598 km), 2nd: 南山 ( 6.289 km)
合歡山北峰           -> 1st: 大禹嶺             ( 3.536 km), 2nd: 合歡山 ( 4.330 km)
合歡山北峰西北峰        -> 1st: 大禹嶺             ( 3.536 km), 2nd: 合歡山 ( 4.330 km)
白姑大山            -> 1st: 中橫21.6k         ( 6.474 km), 2nd: 大雪山埡口 (10.846 km)
北大武山            -> 1st: 瑪家              ( 9.867 km), 2nd: 舊泰武 (10.805 km)
志佳陽大山           -> 1st: 雪山圈谷            ( 3.406 km), 2nd: 雪山東峰 ( 3.829 km)
雪山東峰            -> 1st: 雪山東峰            ( 0.337 km), 2nd: 雪山圈谷 ( 3.611 km)
塔關山             -> 1st: 向陽              ( 4.587 km), 2nd: 復興 (14.209 km)
關山嶺山            -> 1st: 向陽              ( 3.704 km), 2nd: 復興 (16.579 km)
郡大山             -> 1st: 玉山風口            (19.741 km), 2nd: 信義 (22.771 km)
玉山前峰           